In [1]:
import utils
from IPython.display import clear_output
import networkx
import pandas as pd

In [2]:
# define the hyper parameters
NETWORK_FOLDER = "../data/NetworksFromUp-InBetween1/"
DRUG_TARGETS = "../data/DrugTargets.txt"

In [3]:
# open the file with the drug-targets
with open(DRUG_TARGETS, "r") as input_file:
    drug_targets = [line.rstrip() for line in input_file.readlines()]
    
print(f"The {len(drug_targets)} drug-targets have been read successfully.")

The 458 drug-targets have been read successfully.


In [4]:
# read the patient IDs
print(f"The {len(utils.patient_ids)} patient IDs have been read successfully.")

The 151 patient IDs have been read successfully.


In [5]:
# retrieve for each case, the specific proteins data from dictionaires as well as from the coresponding networks and save them
output_object_list = []
index = 0

for patient in utils.patient_ids:

    print(f"{index + 1}/{len(utils.patient_ids)}")  
    index += 1
    
    down_proteins = utils.down_dict.get(patient,[])
    up_proteins = utils.up_dict.get(patient,[])
    mutations = utils.mutation_t.get(patient, [])
    network = networkx.read_edgelist(NETWORK_FOLDER+f"{patient}.txt", delimiter = ";", create_using = networkx.DiGraph)
    
    output_object = {
        "Index": index,
        "Patient ID": patient,
        "Patient Down Proteins": len(down_proteins),
        "Patient Up Proteins": len(up_proteins),
        "Patient Mutations":len(mutations),
        "Network Proteins": network.number_of_nodes(),
        "Network Interactions": network.number_of_edges(),
        "Network Drug-Targets": len([node for node in network.nodes() if node in drug_targets]),
        "Network Down Proteins": len([node for node in network.nodes() if node in down_proteins]),
        "Network Up Proteins": len([node for node in network.nodes() if node in up_proteins]),
        "Network Mutations": len([node for node in network.nodes() if node in mutations])
    }
    
    output_object_list.append(output_object)
    clear_output(wait = True)
    
print(f"The output object list has been defined successfully.")

The output object list has been defined successfully.


In [6]:
# save the dataframe
dataframe = pd.DataFrame(output_object_list)
dataframe.to_excel("../data/networks_statistics.xlsx", index = False, engine = 'xlsxwriter')
  
print(f"The dataframe has been written successfully.")

The dataframe has been written successfully.


In [7]:
# calculate the number of drugs whose drug targets were included in at least one analysis
found_targets = set()
networks = {}

for patient_id in utils.patient_ids:
    G = networkx.read_edgelist(NETWORK_FOLDER+f"{patient_id}.txt", delimiter=";", create_using=networkx.DiGraph)
    networks[patient_id] = G
    found_targets.update(set(G.nodes) & set(drug_targets))

gene_table = pd.read_excel("../data/gene_drugs.xlsx")
drug_table = pd.read_excel("../data/atc_l_drugs.xlsx")

filtered_genes = gene_table[gene_table["Gene"].isin(found_targets)]
valid_drug_ids = set(drug_table["ID"])

expanded = (
    filtered_genes["Drug"]
    .str.split(";")
    .explode()
    .dropna()
)

unique_count = expanded[expanded.isin(valid_drug_ids)].nunique()

print(unique_count)

406
